# Pre-M0.6 — IQ Time Traces and Constellations

## Unit Objective

In this unit you will learn two fundamental ways to **visualize** IQ signals:

1. **Time traces** — plotting `I[t]` and `Q[t]` separately against a time axis.
2. **Constellation diagrams** — plotting `I` against `Q` on a 2D plane.

These visualizations are the *eyes* of any IQ-signal engineer: they reveal signal structure, modulation type, and anomalies that numbers alone often hide.

## What You Should Learn

- How to generate a time-axis vector from sample count and sample rate.
- How to plot `I` and `Q` vs. time (time traces) using `matplotlib.pyplot`.
- How to create a constellation diagram (I vs. Q scatter).
- What each plot style reveals about the signal (frequency, phase, modulation).
- How to compare two different signals visually side by side.

## IQ Data Used

All signals follow the canonical IQ layout:

```python
X.shape == (N, 2, L)
```

| Axis | Meaning           |
|------|-------------------|
|  0   | Examples          |
|  1   | I/Q (0 = I, 1 = Q) |
|  2   | Time samples (L)  |

`dtype = np.float32`, `SEED = 42`.

## Imports

Only the allowed libraries for this unit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Constants and Global Settings

In [ ]:
SEED = 42
SAMPLE_RATE = 1000.0   # samples per second
N_SIGNALS = 4          # number of example signals
L = 256                # time samples per signal
rng = np.random.default_rng(SEED)

---

## 1  Guided Development — Time Traces

A **time trace** plots each component of the IQ signal over the sample index (or real time).
This shows *how* the in-phase and quadrature components evolve as the signal progresses.

### Step 1 — Build a synthetic IQ signal

We construct a single-tone IQ signal:
$$
I[n] = A\cos(2\pi f n / f_s), \quad
Q[n] = A\sin(2\pi f n / f_s)
$$

In [ ]:
def make_single_tone(frequency, amplitude, num_samples, sample_rate, rng):
    """Return X of shape (1, 2, num_samples) — one single-tone IQ signal."""
    n = np.arange(num_samples, dtype=np.float32)
    phase = 2.0 * np.pi * frequency * n / sample_rate
    I = amplitude * np.cos(phase).astype(np.float32)
    Q = amplitude * np.sin(phase).astype(np.float32)
    X = np.stack([I, Q], axis=0)          # (2, L)
    return X[np.newaxis, ...]              # (1, 2, L)

X_tone = make_single_tone(
    frequency=50.0,
    amplitude=1.0,
    num_samples=L,
    sample_rate=SAMPLE_RATE,
    rng=rng
)
print(f"X_tone.shape = {X_tone.shape}  dtype = {X_tone.dtype}")

### Step 2 — Build the time-axis vector

In [ ]:
t = np.arange(X_tone.shape[2], dtype=np.float32) / SAMPLE_RATE
print(f"t.shape = {t.shape}  t range = [{t[0]:.4f}, {t[-1]:.4f}] s")

### Step 3 — Plot I and Q vs. time

In [ ]:
I_tone = X_tone[0, 0, :]  # I channel of first (and only) signal
Q_tone = X_tone[0, 1, :]  # Q channel of first (and only) signal

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, I_tone, label="I", linewidth=1.2)
ax.plot(t, Q_tone, label="Q", linewidth=1.2, linestyle="--")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Amplitude")
ax.set_title("Time Traces — Single-Tone IQ Signal (50 Hz)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Observation:** Both `I` and `Q` are sinusoids at the same frequency but shifted by 90° relative to each other — this is the hallmark of a complex exponential representation.

---

## 2  Guided Development — Constellation Diagram

A **constellation diagram** ignores the time axis entirely and instead plots `I` on the x-axis and `Q` on the y-axis.
It shows the *trajectory* of the complex signal in the I/Q plane.

### Step 1 — Scatter plot of the single-tone constellation

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(I_tone, Q_tone, linewidth=0.8, alpha=0.7, label="trajectory")
ax.plot(I_tone[0], Q_tone[0], "go", markersize=8, label="start")
ax.plot(I_tone[-1], Q_tone[-1], "r*", markersize=10, label="end")
ax.set_xlabel("I")
ax.set_ylabel("Q")
ax.set_title("Constellation — Single-Tone IQ Signal (50 Hz)")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

**Observation:** The trajectory traces a *circle* of radius `A`. This confirms a single-tone complex sinusoid with constant amplitude.

---

## 3  Small Examples — Comparing Different Signals

Let's generate four different IQ signals and compare their time traces and constellations.

In [ ]:
def make_iq_signals(N, L, sample_rate, rng):
    """Generate N different IQ signals of shape (N, 2, L).
    
    Signals:
      0 — single-tone 50 Hz, amplitude 1.0
      1 — single-tone 120 Hz, amplitude 0.5
      2 — two-tone (30 Hz + 80 Hz)
      3 — noisy single-tone (50 Hz, amplitude 1.0, noise 0.2)
    """
    X = np.zeros((N, 2, L), dtype=np.float32)
    n = np.arange(L, dtype=np.float32)

    # Signal 0: 50 Hz, A=1.0
    f0 = 50.0
    phase0 = 2.0 * np.pi * f0 * n / sample_rate
    X[0, 0, :] = 1.0 * np.cos(phase0)
    X[0, 1, :] = 1.0 * np.sin(phase0)

    # Signal 1: 120 Hz, A=0.5
    f1 = 120.0
    phase1 = 2.0 * np.pi * f1 * n / sample_rate
    X[1, 0, :] = 0.5 * np.cos(phase1)
    X[1, 1, :] = 0.5 * np.sin(phase1)

    # Signal 2: two-tone 30 Hz + 80 Hz
    f2a, f2b = 30.0, 80.0
    phase2a = 2.0 * np.pi * f2a * n / sample_rate
    phase2b = 2.0 * np.pi * f2b * n / sample_rate
    X[2, 0, :] = np.cos(phase2a) + 0.6 * np.cos(phase2b)
    X[2, 1, :] = np.sin(phase2a) + 0.6 * np.sin(phase2b)

    # Signal 3: 50 Hz, A=1.0, additive noise
    noise = rng.standard_normal((2, L)).astype(np.float32) * 0.2
    X[3, 0, :] = 1.0 * np.cos(phase0) + noise[0]
    X[3, 1, :] = 1.0 * np.sin(phase0) + noise[1]

    return X

X_all = make_iq_signals(N_SIGNALS, L, SAMPLE_RATE, rng)
print(f"X_all.shape = {X_all.shape}  dtype = {X_all.dtype}")

### Time traces — all four signals

In [ ]:
labels = ["50 Hz, A=1.0", "120 Hz, A=0.5", "Two-tone (30+80 Hz)", "50 Hz + noise"]

fig, axes = plt.subplots(4, 1, figsize=(8, 8), sharex=True)

for i in range(N_SIGNALS):
    axes[i].plot(t, X_all[i, 0, :], label="I", linewidth=0.9)
    axes[i].plot(t, X_all[i, 1, :], label="Q", linewidth=0.9, linestyle="--")
    axes[i].set_ylabel("Amplitude")
    axes[i].set_title(f"Signal {i}: {labels[i]}")
    axes[i].legend(loc="upper right", fontsize=8)
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel("Time [s]")
plt.tight_layout()
plt.show()

### Constellation diagrams — all four signals

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i in range(N_SIGNALS):
    ax = axes[i]
    ax.plot(X_all[i, 0, :], X_all[i, 1, :], linewidth=0.6, alpha=0.7)
    ax.set_xlabel("I")
    ax.set_ylabel("Q")
    ax.set_title(f"Signal {i}: {labels[i]}")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### What each visualization reveals

| Signal | Time Trace Reveals | Constellation Reveals |
|--------|--------------------|-----------------------|
| 0 (50 Hz, A=1) | Two sinusoids, 90° phase shift, constant frequency | Perfect circle, radius = 1 |
| 1 (120 Hz, A=0.5) | Higher frequency, half amplitude | Smaller circle, radius = 0.5 |
| 2 (two-tone) | Complex waveform, non-sinusoidal | Lissajous-like rosette pattern |
| 3 (noisy) | Sine waves with visible jitter | Fuzzy circle (noise scatters points) |

---

## 4  Student Exercises

Complete each exercise **before** continuing.

### Exercise 1 — Build a frequency-sweep signal

Create a signal whose frequency increases linearly from 10 Hz to 100 Hz over `L` samples.
Plot both its time trace and constellation.

**Hint:** Use `np.cumsum` or a linear phase ramp.

In [ ]:
# STUDENT ATTEMPT — Exercise 1
# Create a frequency-sweep (chirp) IQ signal
# Hint: phase = integral of instantaneous frequency
#       freq[n] = f_start + (f_end - f_start) * n / (L - 1)
#       phase[n] = cumsum(2*pi*freq[n]/fs)

# YOUR CODE HERE


### Exercise 2 — Dual-constellation overlay

Using `X_all`, plot the constellations of signals 0 and 3 (clean vs. noisy) **on the same axes**.
Use different colors and a legend.

In [ ]:
# STUDENT ATTEMPT — Exercise 2
# Overlay constellations of X_all[0] and X_all[3]

# YOUR CODE HERE


### Exercise 3 — Read a constellation

Given the constellation of signal 2 (two-tone), estimate how many distinct frequency components contribute to the signal.
Explain how the constellation shape hints at this.

Write your answer in the Markdown cell below.

**Your answer here (Exercise 3):**





---

## OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

The cell below contains the solution to Exercise 1. **Do not look at it before attempting the exercise.**

In [ ]:
# OPTIONAL SOLUTION — Exercise 1
f_start, f_end = 10.0, 100.0
n_idx = np.arange(L, dtype=np.float32)
freq_inst = f_start + (f_end - f_start) * n_idx / (L - 1)
phase_chirp = np.cumsum(2.0 * np.pi * freq_inst / SAMPLE_RATE).astype(np.float32)

X_chirp = np.zeros((1, 2, L), dtype=np.float32)
X_chirp[0, 0, :] = np.cos(phase_chirp)
X_chirp[0, 1, :] = np.sin(phase_chirp)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(t, X_chirp[0, 0, :], label="I")
ax1.plot(t, X_chirp[0, 1, :], label="Q", linestyle="--")
ax1.set_xlabel("Time [s]")
ax1.set_ylabel("Amplitude")
ax1.set_title("Time Trace — Chirp 10→100 Hz")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(X_chirp[0, 0, :], X_chirp[0, 1, :], linewidth=0.6)
ax2.set_xlabel("I")
ax2.set_ylabel("Q")
ax2.set_title("Constellation — Chirp 10→100 Hz")
ax2.set_aspect("equal")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"X_chirp.shape = {X_chirp.shape}")

In [ ]:
# OPTIONAL SOLUTION — Exercise 2
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(X_all[0, 0, :], X_all[0, 1, :], linewidth=0.6, alpha=0.8, label="Signal 0 (clean)")
ax.plot(X_all[3, 0, :], X_all[3, 1, :], linewidth=0.6, alpha=0.5, label="Signal 3 (noisy)")
ax.set_xlabel("I")
ax.set_ylabel("Q")
ax.set_title("Overlay — Clean vs. Noisy Constellation")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

---

## PASS CRITERION CHALLENGE

Before proceeding, make sure you have:
- Completed all three exercises (or given your best attempt).
- Understood the difference between time traces and constellation diagrams.

The three **Pass Criterion (PC)** gates below formally verify your understanding.
You must pass **all three gates** to receive a PASS status.

---

## PC-1 — INDEPENDENT AXIS EXPLANATION

### STUDENT ATTEMPT

Answer **all six questions** below in the Markdown cell.
These questions probe your understanding of the axis semantics in `X.shape == (N, 2, L)`.

**Question 1:** What does axis 0 represent in `X.shape == (N, 2, L)`?

**Your answer:**



---

**Question 2:** What does axis 1 represent, and what do the values 0 and 1 mean?

**Your answer:**



---

**Question 3:** What does axis 2 represent?

**Your answer:**



---

**Question 4:** To extract the I-channel of the *third* signal, what is the correct indexing expression?

**Your answer:**



---

**Question 5:** If `X.shape == (10, 2, 512)`, how many total I/Q samples does the dataset contain?

**Your answer:**



---

**Question 6:** In a time trace plot, which two axes of `X` are used? In a constellation plot, which two are used?

**Your answer:**





In [ ]:
# Student/instructor: change to True after verifying all 6 answers are correct
AXES_EXPLANATION_VERIFIED = False  # SET TO True AFTER MANUAL VERIFICATION
print(f"AXES_EXPLANATION_VERIFIED = {AXES_EXPLANATION_VERIFIED}")

---

## OPTIONAL SOLUTION — PC-1 (REVEAL ONLY AFTER ATTEMPT)

| # | Question | Answer |
|---|----------|--------|
| 1 | Axis 0 | Number of independent IQ signals (examples) |
| 2 | Axis 1 | I/Q component: 0 = In-phase (I), 1 = Quadrature (Q) |
| 3 | Axis 2 | Time samples (L) |
| 4 | I of 3rd signal | `X[2, 0, :]` |
| 5 | Total I/Q samples | 10 × 2 × 512 = 10,240 |
| 6 | Axes used | Time trace: axes (2) vs. (1) per signal; Constellation: axis (0) vs. axis (1) for one signal, or equivalently, we use `X[i, 0, :]` (axis 2) and `X[i, 1, :]` (axis 2) — i.e., the I and Q vectors along axis 2 |

---

## PC-2 — INJECTED AXIS SWAP

In this gate, we **deliberately inject** an axis-ordering error and ask you to diagnose and fix it.

In [ ]:
X = make_iq_signals(N_SIGNALS, L, SAMPLE_RATE, rng)
print(f"X.shape = {X.shape}  (expected (N, 2, L))")

# --- INJECTED ERROR: transposed axes 1 and 2 ---
X_swapped = np.transpose(X, (0, 2, 1))
print(f"X_swapped.shape = {X_swapped.shape}  (WRONG — I/Q axis is now axis 2)")

### STUDENT ATTEMPT — PC-2

1. **Inspect the shape** of `X_swapped` and `X`.
2. **Identify** which axis now holds the I/Q components in `X_swapped`.
3. **Explain** why `X_swapped` is wrong (what would happen if you tried to plot a constellation from it?).
4. **Write the correction** in the code cell below.
5. **Produce** `X_fixed` and **verify** it matches `X` exactly.

**Your explanation (written before code):**





In [ ]:
# STUDENT ATTEMPT — PC-2: Fix the axis swap

# YOUR CODE HERE
# X_fixed = ...


---

## OPTIONAL SOLUTION — PC-2 (REVEAL ONLY AFTER ATTEMPT)

In [ ]:
# OPTIONAL SOLUTION — PC-2
# The swap transposed axes 1 and 2, so to undo it we transpose back:
X_fixed = np.transpose(X_swapped, (0, 2, 1))

print(f"X_fixed.shape = {X_fixed.shape}  (should be (N, 2, L))")
print(f"X_fixed.dtype = {X_fixed.dtype}")

### Automatic Validation — PC-2

In [ ]:
axis_swap_corrected = (
    X_fixed.shape == X.shape
    and X_fixed.dtype == X.dtype
    and np.array_equal(X_fixed, X)
)
assert X_fixed.shape == X.shape, f"Shape mismatch: {X_fixed.shape} != {X.shape}"
assert X_fixed.dtype == X.dtype, f"dtype mismatch: {X_fixed.dtype} != {X.dtype}"
assert np.array_equal(X_fixed, X), "Arrays differ — correction failed"
print(f"axis_swap_corrected = {axis_swap_corrected}")
print("PC-2 PASSED")

---

## PC-3 — IQ POWER AGREEMENT

Two independent ways to compute the **mean power** of an IQ signal:

- **Method A (I²+Q²):** `P_iq = mean(I² + Q²)`
- **Method B (complex):** `z = I + jQ; P_complex = mean(|z|²)`

These MUST agree to floating-point tolerance.

In [ ]:
# Work with signal 0 for this gate
I_signal = X[0, 0, :]
Q_signal = X[0, 1, :]

### STUDENT ATTEMPT — PC-3

Implement **Method A** and **Method B** independently, then validate.

In [ ]:
# STUDENT ATTEMPT — PC-3

# Method A: P_iq = mean(I**2 + Q**2)
# YOUR CODE HERE
# P_iq = ...

# Method B: z = I + 1j*Q; P_complex = mean(|z|**2)
# YOUR CODE HERE
# P_complex = ...


---

## OPTIONAL SOLUTION — PC-3 (REVEAL ONLY AFTER ATTEMPT)

In [ ]:
# OPTIONAL SOLUTION — PC-3

# Method A
P_iq = np.mean(I_signal**2 + Q_signal**2)

# Method B
z = I_signal + 1j * Q_signal
P_complex = np.mean(np.abs(z)**2)

print(f"P_iq      = {P_iq}")
print(f"P_complex = {P_complex}")

### Automatic Validation — PC-3

In [ ]:
POWER_RTOL = 1e-5
POWER_ATOL = 1e-7

power_consistency = np.allclose(P_iq, P_complex, rtol=POWER_RTOL, atol=POWER_ATOL)
assert power_consistency, (
    f"Power mismatch: P_iq={P_iq}, P_complex={P_complex}, "
    f"diff={abs(P_iq - P_complex)}"
)
print(f"power_consistency = {power_consistency}")
print(f"|P_iq - P_complex| = {abs(P_iq - P_complex):.2e}")
print("PC-3 PASSED")

---

## Manual Evaluation — Axis Explanation (PC-1)

If you have not already done so, review your six answers in the PC-1 section above.
The instructor (or you, after self-review) should set `AXES_EXPLANATION_VERIFIED = True` in the code cell provided in PC-1.

In [ ]:
if not AXES_EXPLANATION_VERIFIED:
    print("WARNING: AXES_EXPLANATION_VERIFIED is still False.")
    print("Please review your PC-1 answers and set it to True.")
else:
    print("PC-1 axis explanation: VERIFIED")

---

## PASS CRITERION GATE

Collect the results from all three gates and display the final status.

In [ ]:
pc1_pass = AXES_EXPLANATION_VERIFIED
pc2_pass = axis_swap_corrected
pc3_pass = power_consistency

print("=" * 50)
print("PASS CRITERION GATE — Pre-M0.6")
print("=" * 50)
print(f"PC-1  Axis Explanation     : {'PASS' if pc1_pass else 'WAIT'}")
print(f"PC-2  Axis Swap Correction : {'PASS' if pc2_pass else 'WAIT'}")
print(f"PC-3  IQ Power Agreement   : {'PASS' if pc3_pass else 'WAIT'}")
print("-" * 50)

all_pass = pc1_pass and pc2_pass and pc3_pass
final_status = "PASS" if all_pass else "WAIT"
print(f"\nPRE-M0.6 FINAL STATUS: {final_status}")